In [ ]:
# %%
# SECTION 1: Imports
# -------------------
# Loads all required libraries:
#   - requests: fetches the raw HTML content from a URL
#   - BeautifulSoup: parses the HTML so we can search and extract elements
#   - csv: writes extracted data to a CSV file
#   - sys: used for clean error exits

import csv
import sys

import requests
from bs4 import BeautifulSoup

print("Imports loaded successfully.")

In [ ]:
# %%
# SECTION 2: Configuration
# -------------------------
# Set your scrape target and options here before running any sections below.
#
# url         - The page you want to scrape
# tag         - The HTML tag to search for (e.g. 'td', 'li', 'span')
# class_name  - CSS class to filter by (set to None to skip class filtering)
# col_index   - For table scraping: which column index to pull (set to None to skip)
# scrape_list - Set to True to scrape all <li> elements instead
# output_path - Where to save the resulting CSV

url         = 'https://streetzipcode.net/canada-alberta-fort-saskatchewan-postal-codes'
tag         = 'td'
class_name  = 'text-center'   # e.g. 'text-center', or None
col_index   = None             # e.g. 2 for the 3rd column, or None
scrape_list = False            # True to scrape <li> elements
output_path = 'output.csv'

print(f"Config ready. Target URL: {url}")

In [ ]:
# %%
# SECTION 3: fetch_page()
# ------------------------
# Sends an HTTP GET request to the configured URL and returns the raw page content.
# Sets a User-Agent header so the request looks like a normal browser visit.
# Raises an error and stops if the request fails (e.g. 404, timeout, no connection).
# Run this section first — all other sections depend on the 'page_content' variable it produces.

def fetch_page(url: str) -> bytes:
    headers = {'User-Agent': 'Mozilla/5.0'}
    response = requests.get(url, headers=headers, timeout=10)
    response.raise_for_status()
    return response.content

try:
    page_content = fetch_page(url)
    print(f"Page fetched successfully. ({len(page_content)} bytes)")
except requests.RequestException as e:
    print(f"Error fetching page: {e}")
    page_content = None

In [ ]:
# %%
# SECTION 4: Parse HTML
# ----------------------
# Takes the raw HTML bytes from fetch_page() and parses them into a navigable
# BeautifulSoup object ('soup'). This is required before any of the extraction
# functions below can run. Prints a short preview of the parsed content.

if page_content:
    soup = BeautifulSoup(page_content, 'html.parser')
    print("HTML parsed successfully.")
    print("Preview (first 500 chars):")
    print(soup.get_text()[:500])
else:
    print("No page content to parse. Run Section 3 first.")

In [ ]:
# %%
# SECTION 5: scrape_by_class()
# -----------------------------
# Finds all HTML elements matching the configured 'tag' and 'class_name' and
# extracts their inner text. This is the most common mode — useful when the
# target data lives inside elements with a specific CSS class (e.g. a table cell
# with class='text-center'). Skips any blank elements.
# Results are stored in 'data' and printed for review.

def scrape_by_class(soup, tag: str, class_name: str) -> list:
    elements = soup.find_all(tag, class_=class_name)
    return [el.get_text(strip=True) for el in elements if el.get_text(strip=True)]

if class_name:
    data = scrape_by_class(soup, tag, class_name)
    print(f"Found {len(data)} items via class filter '{class_name}':")
    print(data[:20])
else:
    print("class_name is not set. Update Section 2 config to use this mode.")

In [ ]:
# %%
# SECTION 6: scrape_table_column()
# ---------------------------------
# Iterates over every <tr> (table row) on the page and pulls out the cell at
# position 'col_index'. Useful when data is in a plain HTML table without
# distinguishing CSS classes (e.g. geonames.org-style tables where the postal
# code is always in the 3rd column). Skips rows that are too short.
# Results are stored in 'data' and printed for review.

def scrape_table_column(soup, col_index: int) -> list:
    results = []
    for row in soup.find_all('tr'):
        cells = row.find_all('td')
        if len(cells) > col_index:
            text = cells[col_index].get_text(strip=True)
            if text:
                results.append(text)
    return results

if col_index is not None:
    data = scrape_table_column(soup, col_index)
    print(f"Found {len(data)} items from column index {col_index}:")
    print(data[:20])
else:
    print("col_index is not set. Update Section 2 config to use this mode.")

In [ ]:
# %%
# SECTION 7: scrape_list_items()
# --------------------------------
# Finds all <li> elements on the page and extracts their text. Useful for pages
# that present data as bullet lists or navigation menus (e.g. a product catalog
# or a list of search results). Skips blank items.
# Results are stored in 'data' and printed for review.

def scrape_list_items(soup) -> list:
    return [li.get_text(strip=True) for li in soup.find_all('li') if li.get_text(strip=True)]

if scrape_list:
    data = scrape_list_items(soup)
    print(f"Found {len(data)} list items:")
    print(data[:20])
else:
    print("scrape_list is False. Set it to True in Section 2 config to use this mode.")

In [ ]:
# %%
# SECTION 8: Auto-select and run extraction
# ------------------------------------------
# Looks at your Section 2 config and automatically runs the right extraction
# function: list items > table column > class filter > generic tag fallback.
# Use this section for a one-shot run once you know your config is correct.
# The results are stored in 'data' for use in the CSV export section below.

if scrape_list:
    data = scrape_list_items(soup)
    mode = 'list items'
elif col_index is not None:
    data = scrape_table_column(soup, col_index)
    mode = f'table column {col_index}'
elif class_name:
    data = scrape_by_class(soup, tag, class_name)
    mode = f'<{tag} class="{class_name}">'
else:
    data = [el.get_text(strip=True) for el in soup.find_all(tag) if el.get_text(strip=True)]
    mode = f'all <{tag}> elements'

print(f"Mode: {mode}")
print(f"Total items found: {len(data)}")
print("Preview:", data[:10])

In [ ]:
# %%
# SECTION 9: write_csv()
# -----------------------
# Takes the 'data' list produced by any of the extraction sections and writes it
# to a CSV file at the path set in 'output_path' (Section 2 config). Each item
# gets its own row under a 'value' header. Prints a confirmation with the row count.
# Run this section last, after verifying the preview in Section 8 looks correct.

def write_csv(data: list, output_path: str) -> None:
    with open(output_path, 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(['value'])
        for item in data:
            writer.writerow([item])
    print(f"Wrote {len(data)} rows to '{output_path}'")

if data:
    write_csv(data, output_path)
else:
    print("No data to write. Run an extraction section first.")